## Geometric characterisations of JEPA displacement field

Basic geometric analysis pipeline on the three displacement vectors:
- $\Delta$ (P-C): predicted displacement (what the model expects to change)
- Error ($P-T$): prediction error (what the model got wrong)
- observed trajectory ($T-C$): observed trajectory (what actually changed)

Vector Norms, PCA on each vector, Shared-Basis Decomposition, UMAP on $z_{ctx}$

In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
from matplotlib import pyplot as plt
from scipy.sparse.linalg import eigsh
from src.utils.io import EXPERIMENTS_DIR
from src.utils.seed import load_seed, set_global_seed
from src.analysis.geometry import (
    fit_pca, get_pca_stats, 
    fit_tform_umap_2d,
    fit_tform_phate_2d, fit_phate,
    compute_icc,
    # cosine_sim_matrix, cosine_dist_matrix,
    # find_divergent_pairs, regress_divergence,
    # divergence_variance_comparison,
)
# from src.analysis.plotting import (
#     show_or_savefig,
#     plot_three_vector_norm_comparison,
#     plot_pca_scree_v_mp_upper, plot_cum_var,
#     plot_pca1_vs_pca2,
#     plot_shared_basis_decomposition,
#     plot_context_umap,
#     plot_icc_bar, plot_spaghetti,
#     plot_scatter_pairwise_dvg_scatter,
#     plot_divergence_pc_projections,
#     plot_variance_comparison,
# )

In [ ]:
# -- Config Settings --
MODEL_TAG        = "test_01"
EMB_NAME         = "embeddings_40.npz"
TOP_K_PCA        = 10
UMAP_NN          = 10
PHATE_NN         = 10
SAVE_FIGS        = False
SAVE_DATA        = True

exp_dir  = EXPERIMENTS_DIR / MODEL_TAG
set_global_seed(load_seed(exp_dir))

emb_path = exp_dir / "embeddings" / EMB_NAME
geo_dir  = exp_dir / "geometry"
fig_dir  = geo_dir / "figures"

def _sp(name: str):
    return fig_dir / name if SAVE_FIGS else None

In [ ]:
# -- Load embeddings --
npz = np.load(emb_path, allow_pickle=True)
z_context      = npz["z_context"]
z_pred         = npz["z_pred"]
z_target       = npz["z_target"]
labels         = npz["labels"]
subject_ids    = npz["subject_ids"]
mask_positions = npz["mask_positions"]

# -- Compute (or load) vectors --
delta         = npz["delta"]         if "delta"         in npz else z_pred - z_context
pred_error    = npz["pred_error"]    if "pred_error"    in npz else z_pred - z_target
observed_traj = npz["observed_traj"] if "observed_traj" in npz else z_target - z_context

N, D = z_context.shape
print(f"Loaded {emb_path.name}:  N={N}  D={D}")
print(f" ||Δ||   mean={np.linalg.norm(delta, axis=-1).mean():.5f}")
print(f" ||P-T|| mean={np.linalg.norm(pred_error, axis=-1).mean():.5f}")
print(f" ||T-C|| mean={np.linalg.norm(observed_traj, axis=-1).mean():.5f}")

**Norm Overview**

In [ ]:
from src.analysis.plotting import plot_three_vector_norm_comparison
plot_three_vector_norm_comparison(
    delta, pred_error, observed_traj, labels,
    save_path=_sp("norm_comparison.png"),
)

### PCA on each vector

In [ ]:
from src.analysis.plotting import (
    plot_pca_scree_v_mp_upper, 
    plot_cum_var,
    plot_pca1_vs_pca2)

In [ ]:
pca_objects = {}     # name -> PCA object
pca_projections = {} # name -> (N, D) full projections

for (name, vec, display) in [
    ("delta", delta, "(P-C)"),
    ("pred_error", pred_error, "(P-T)"),
    ("observed_traj", observed_traj, "(T-C)"),
]:
    print(f"\n{'='*60}\n  PCA on {display}\n{'='*60}")
    
    pca, proj_full, proj_topk = fit_pca(vec, k=TOP_K_PCA)
    pca_stats = get_pca_stats(pca, k=TOP_K_PCA, n_samples=N)
    
    pca_objects[name] = pca
    pca_projections[name] = proj_full
    print(f"  Signal components (MP): {pca_stats['n_signal_components']}")
    print(f"  Effective dim:          {pca_stats['effective_dimensionality']:.2f}")
    print(f"  Top-{TOP_K_PCA} explained:    {pca_stats['top_k_explained_variance']:.2%}")
    print(f"  90% at {pca_stats['components_for_90pct']} PCs, "
          f"  95% at {pca_stats['components_for_95pct']} PCs")

    # Scree plot with Markenko-Pastur comparison line
    evs = np.array(pca_stats["eigenvalues_all"])
    plot_pca_scree_v_mp_upper(
        evs, pca_stats["mp_upper_bound"], pca_stats["n_signal_components"],
        D, pca_stats["effective_dimensionality"],
        vector_name=display,
        save_path=_sp(f"{name}_scree.png"),
    )
    # Cumulative variance
    plot_cum_var(
        pca.explained_variance_ratio_, D, pca_stats["effective_dimensionality"],
        save_path=_sp(f"{name}_cumvar.png"),
    )
    # PC1 vs PC2 by label
    plot_pca1_vs_pca2(
        proj_full, labels, pca.explained_variance_ratio_,
        save_path=_sp(f"{name}_pc1_pc2.png"),
    )

    # Save projections and stats
    if SAVE_DATA:
        geo_dir.mkdir(parents=True, exist_ok=True)
        np.save(geo_dir / f"{name}_projections.npy", proj_full)
        with open(geo_dir / f"{name}_pca_stats.json", "w") as f:
            json.dump({k: v for k, v in pca_stats.items()
                       if k != "eigenvalues_all"}, f, indent=2)

### Shared-Basis Decomposition

PCA on observed trajectory (T-C) as the reference basis. Project all three vectors onto it, then compute per-axis capture ratio: $\displaystyle\frac{1 - var(P-T)}{var(T-C)}$.

This is a headline thesis figure.

In [ ]:
ref_pca = pca_objects["observed_traj"]
ref_components = ref_pca.components_   # (D, D)

# Project all three vectors onto the reference basis
proj_ot = pca_projections["observed_traj"]   # already in this basis
proj_d  = delta @ ref_components.T
proj_pe = pred_error @ ref_components.T

var_ot = proj_ot.var(axis=0)
var_d  = proj_d.var(axis=0)
var_pe = proj_pe.var(axis=0)

In [ ]:
from src.analysis.plotting import plot_shared_basis_decomposition
plot_shared_basis_decomposition(
    var_ot, var_d, var_pe,
    ref_pca.explained_variance_ratio_,
    top_k=TOP_K_PCA,
    save_path=_sp("shared_basis_decomposition.png"),
)

In [ ]:
# -- Per-axis capture ratio table --
with np.errstate(divide="ignore", invalid="ignore"):
    capture = np.where(
        var_ot[:TOP_K_PCA] > 1e-12, 1.0 - var_pe[:TOP_K_PCA] / var_ot[:TOP_K_PCA], 
                                    np.nan)

print(f"\n{'PC':<6} {'var(T-C)':>10} {'var(P-C)':>10} {'var(P-T)':>10} {'capture':>10}")
print("-" * 50)
for i in range(TOP_K_PCA):
    print(f"PC{i+1:<4} {var_ot[i]:>10.4f} {var_d[i]:>10.4f} {var_pe[i]:>10.4f} {capture[i]:>9.1%}")

if SAVE_DATA:
    decomp = {
        "variances_observed": var_ot[:TOP_K_PCA].tolist(),
        "variances_delta": var_d[:TOP_K_PCA].tolist(),
        "variances_pred_error": var_pe[:TOP_K_PCA].tolist(),
        "capture_ratio": [float(c) if not np.isnan(c) else None for c in capture],
        "explained_variance_ratio": ref_pca.explained_variance_ratio_[:TOP_K_PCA].tolist(),
    }
    with open(geo_dir / "shared_basis_decomposition.json", "w") as f:
        json.dump(decomp, f, indent=2)

### UMAP on $z_{ctx}$

*Uniform Manifold Approximation and Projection for Dimension Reduction*

UMAP constructs a fuzzy weighted graph approximating the topological structure of the high-dimensional point cloud, then finds a low-dimensional embedding that preserves the topology by balancing attraction between nearby points against repulsion between distant points. 

UMAP excels at revealing *discrete cluster structure* and *local density variation*. The space between and shapes of clusters is less meaningful as continuous gradients in the data get increasingly discretized, however UMAP is the primary substrate for HDBSCAN cluster enrichment analysis -  the sharp cluster boundaries make it the natural choice for identifying discrete patient subpopulations and testing whether each cluster has a coherent clinical signature (The clusters that emerge from UMAP space define the neighborhoods against which SAE features are cross-referenced in. a SAE feature that concentrates in a single UMAP cluster is capturing something the nonlinear topology already identifies, while a SAE feature that cuts across multiple clusters is finding finer-grained structure that UMAP's discrete topology doesn't resolve.)

In [ ]:
umap_emb = fit_tform_umap_2d(z_context, metric="euclidean", n_neighbors=UMAP_NN)

In [ ]:
from src.analysis.plotting import plot_context_umap
plot_context_umap(umap_emb, labels, save_path=_sp("context_umap.png"))

In [ ]:
# -- Secondary, maybe remove --
from src.analysis.plotting import plot_pca_scree_v_mp_upper

mp_upper = pca_stats['mp_upper_bound']
n_signal = pca_stats['n_signal_components']
eff_dim = pca_stats['effective_dimensionality']
eigenvalues  = pca.explained_variance_

plot_pca_scree_v_mp_upper(
    eigenvalues, mp_upper, n_signal, D=TOP_K_PCA, eff_dim=eff_dim, show=True,
    save_path=_sp("context_scree.png")
)

In [ ]:
if SAVE_DATA:
    geo_dir.mkdir(parents=True, exist_ok=True)
    np.save(geo_dir / "context_umap.npy", umap_emb)

#### PHATE on $z_{ctx}$
*Potential of Heat-diffusion for Affinity-based Trajectory Embedding*

PHATE builds a diffusion operator (transition matrix) over the point cloud. It encodes the probability of reaching any point from any other point via a random walk on the data manifold, then applies a potential distance transform to convert the probabilities into distances that preserve both neighborhood structure and global trajectory geometry. The result is a 2D embedding where euclidean distance approximates diffusion distance on the underlying manifold. 

This makes PHATE particularly well-suited for data with trajectory-like structure - gradual clinical transitions between patient states are rendered as smooth, continuous paths in the embedding rather than being shattered into discrete clumps. Where UMAP acts as in identification, PHATE acts to identify state change among the global embedding geometry. Patients that are far apart in PHATE space are genuinely far apart in terms of the dynamics encoded in the latent representation.

In [ ]:
from src.analysis.plotting import plot_umap_phate_comparison, plot_phate_eigen_decomp

In [ ]:
phate_emb = fit_tform_phate_2d(z_context, knn = PHATE_NN)

In [ ]:
plot_umap_phate_comparison(umap_emb, phate_emb, labels, save_path=_sp("umap_phate_comparison"))

In [ ]:
# Extract the diffusion operator from the fitted PHATE object
# (refit to get access to the operator)
phate_reduce = fit_phate(z_context, PHATE_NN, dims=2)
phate_reduce = phate_reduce.diff_op  # sparse (N, N) diffusion operator

# Top eigenvalues of the diffusion operator (descending)
n_eigs = min(z_context.shape[0], phate_reduce.shape[0] - 2)
eigs, _ = eigsh(phate_reduce, k=n_eigs, which="LM")
eigs = np.sort(eigs)[::-1]

In [ ]:
plot_phate_eigen_decomp(eigs, PHATE_NN, n_eigs, save_path=_sp("phate_eigen_decomp.png"))

### Intra-Class Correlation (ICC) on observed_traj projections

Per-PC intraclass correlation across encounter windows within each patient. Similar to 
- High ICC $\rightarrow$ trait-like axis
- low ICC $\rightarrow$ state-like axis

In [ ]:
ot_proj = pca_projections["observed_traj"]
icc_result = compute_icc(ot_proj, subject_ids, top_k=TOP_K_PCA)

# compute_icc doesn't return "method" but plot_icc_bar expects it
icc_result["method"] = "ICC3,1"

print(f"Eligible patients: {icc_result['eligible_patients']}")
print(f"Trait PCs (ICC > {icc_result['trait_threshold']}): {icc_result['trait_pcs']}")
print(f"State PCs (ICC < {icc_result['state_threshold']}): {icc_result['state_pcs']}")
print()
for pc, val in icc_result["icc_per_pc"].items():
    print(f"  {pc}: {val:.3f}" if val is not None else f"  {pc}: N/A")

In [ ]:
from src.analysis.plotting import plot_icc_bar
plot_icc_bar(icc_result, save_path=_sp("icc_bar.png"))

In [ ]:
# -- spaghetti plots for first 2 PCs --

from src.analysis.plotting import plot_spaghetti
for pc_idx in range(min(2, TOP_K_PCA)):
    plot_spaghetti(
        ot_proj, subject_ids, mask_positions,
        pc_idx=pc_idx,
        save_path=_sp(f"spaghetti_pc{pc_idx + 1}.png"),
    )

if SAVE_DATA:
    with open(geo_dir / "observed_traj_icc.json", "w") as f:
        json.dump({k: v for k, v in icc_result.items() if k != "method"}, f, indent=2)